In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict

def dcg_at_k(rels, k):
    rels = np.asfarray(rels)[:k]
    if rels.size == 0:
        return 0.0
    discounts = np.log2(np.arange(2, rels.size + 2))
    return np.sum(rels / discounts)

def ndcg_at_k(binary_rels, k):
    # binary_rels: list/array of 0/1 relevance in ranked order
    dcg = dcg_at_k(binary_rels, k)
    ideal = dcg_at_k(sorted(binary_rels, reverse=True), k)
    return 0.0 if ideal == 0 else float(dcg / ideal)

def recall_at_k(recommended, relevant_set, k):
    if not relevant_set:
        return np.nan
    rec_k = recommended[:k]
    hits = len(set(rec_k) & relevant_set)
    return hits / len(relevant_set)

def precision_at_k(recommended, relevant_set, k):
    rec_k = recommended[:k]
    if k == 0:
        return np.nan
    hits = len(set(rec_k) & relevant_set)
    return hits / k

def average_precision_at_k(recommended, relevant_set, k):
    if not relevant_set:
        return np.nan
    rec_k = recommended[:k]
    hits = 0
    sum_prec = 0.0
    for i, item in enumerate(rec_k, start=1):
        if item in relevant_set:
            hits += 1
            sum_prec += hits / i
    return sum_prec / min(len(relevant_set), k)

In [ ]:
# === Ratings file (your schema) ===
ratings_path = "../Dataset/Raw/large_user_ratings.csv"
ratings_df = pd.read_csv(ratings_path)

USER_COL = "Username"
ITEM_COL = "BGGId"
RATING_COL = "Rating"

print("ratings_df shape:", ratings_df.shape)
print("columns:", ratings_df.columns.tolist())
ratings_df.head()

In [ ]:
# basic cleaning
ratings_df = ratings_df.dropna(subset=[USER_COL, ITEM_COL, RATING_COL]).copy()
ratings_df[ITEM_COL] = ratings_df[ITEM_COL].astype(int)
ratings_df[RATING_COL] = pd.to_numeric(ratings_df[RATING_COL], errors="coerce")
ratings_df = ratings_df.dropna(subset=[RATING_COL]).copy()

# === relevance threshold (recommended) ===
# 8.0 = strong like (cleaner ground truth, fewer users)
# 7.0 = looser like (more users, noisier)
RELEVANCE_THRESHOLD = 8.0

seen_by_user = ratings_df.groupby(USER_COL)[ITEM_COL].apply(lambda s: set(s.tolist())).to_dict()

relevant_by_user = (
    ratings_df[ratings_df[RATING_COL] >= RELEVANCE_THRESHOLD]
    .groupby(USER_COL)[ITEM_COL]
    .apply(lambda s: set(s.tolist()))
    .to_dict()
)

# users with enough relevant items to split into seeds + holdout
MIN_RELEVANT_FOR_EVAL = 4  # must be >= SEED_SIZE+1 later
eval_users = [u for u, rel in relevant_by_user.items() if len(rel) >= MIN_RELEVANT_FOR_EVAL]

print("Users total:", ratings_df[USER_COL].nunique())
print("Items total:", ratings_df[ITEM_COL].nunique())
print("Eval users:", len(eval_users))
print("Example user relevant count:", len(relevant_by_user[eval_users[0]]) if eval_users else None)

In [ ]:
from pathlib import Path
import joblib
from sklearn.preprocessing import MinMaxScaler

PROCESSED = Path("../Dataset/Processed")

# metadata
df_meta = (
    pd.read_csv("../Dataset/Raw/games_detailed_info.csv", low_memory=False)
      .rename(columns={"primary": "name"})
      .set_index("id")
)

if "name" not in df_meta.columns:
    if "primary" in df_meta.columns:
        df_meta["name"] = df_meta["primary"]
    elif "Name" in df_meta.columns:
        df_meta["name"] = df_meta["Name"]
    else:
        df_meta["name"] = ""

# similarity assets (must exist, same as Notebook 06)
desc_sim = joblib.load(PROCESSED / "desc_topk_csr.joblib")
content_sim = joblib.load(PROCESSED / "content_topk_csr.joblib")
emb_sim = joblib.load(PROCESSED / "emb_topk_csr.joblib")

meta_ids = df_meta.index.to_numpy()
id_to_idx = {int(gid): idx for idx, gid in enumerate(meta_ids)}
idx_to_id = {v: k for k, v in id_to_idx.items()}

# sentiment summary (load once)
sent_path = PROCESSED / "sentiment_summary.csv"
sent_summary = pd.read_csv(sent_path, index_col="id") if sent_path.exists() else None


def topk_from_sim(seed_ids, sim_csr, k=500, weight_strategy="mean"):
    seed_idxs = [id_to_idx[sid] for sid in seed_ids if sid in id_to_idx]
    if not seed_idxs:
        return pd.Series(dtype=float)

    scores = defaultdict(float)
    counts = defaultdict(int)

    for si in seed_idxs:
        row = sim_csr.getrow(si)
        cols = row.indices
        vals = row.data

        for c, v in zip(cols, vals):
            gid = idx_to_id[c]
            if weight_strategy == "max":
                scores[gid] = max(scores[gid], float(v))
            else:
                scores[gid] += float(v)
                counts[gid] += 1

    if weight_strategy != "max":
        for gid in list(scores.keys()):
            scores[gid] = scores[gid] / max(counts[gid], 1)

    for sid in seed_ids:
        scores.pop(sid, None)

    items = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    return pd.Series(dict(items), dtype=float)


def cf_scores_for_seed(seed_game_ids):
    return pd.Series(0.0, index=meta_ids)


def safe_minmax(series, index):
    arr = series.values.astype(float)

    # extra safety: handle all-NaN
    if np.all(np.isnan(arr)):
        return pd.Series(0.0, index=index)

    if np.nanmax(arr) == np.nanmin(arr):
        return pd.Series(0.0, index=index)

    arr = np.nan_to_num(arr, nan=np.nanmean(arr))
    return pd.Series(MinMaxScaler().fit_transform(arr.reshape(-1, 1)).flatten(), index=index)

sent_mean = None
if sent_summary is not None and "avg_sentiment_score" in sent_summary.columns:
    sent_mean = float(sent_summary["avg_sentiment_score"].mean())

def hybrid_recommend(seed_game_ids, alpha=0.4, beta=0.25, gamma=0.25, delta=0.1, topk=20):
    seed_game_ids = [int(x) for x in seed_game_ids]

    cf_series = cf_scores_for_seed(seed_game_ids).reindex(meta_ids).fillna(0.0)
    cf_norm = safe_minmax(cf_series, meta_ids)

    CANDIDATE_K = 800

    desc_series = topk_from_sim(seed_game_ids, desc_sim, k=CANDIDATE_K).reindex(meta_ids).fillna(0)
    content_series = topk_from_sim(seed_game_ids, content_sim, k=CANDIDATE_K).reindex(meta_ids).fillna(0)
    desc_norm = safe_minmax(desc_series, meta_ids)
    content_norm = safe_minmax(content_series, meta_ids)

    emb_series = topk_from_sim(seed_game_ids, emb_sim, k=CANDIDATE_K).reindex(meta_ids).fillna(0)
    emb_norm = safe_minmax(emb_series, meta_ids)
    

    if sent_summary is not None and "avg_sentiment_score" in sent_summary.columns:
        sent = sent_summary["avg_sentiment_score"].reindex(meta_ids)
        sent = sent.fillna(sent_mean)
        sent_norm = safe_minmax(sent, meta_ids)
    else:
        sent_norm = pd.Series(0.0, index=meta_ids)

    text_norm = 0.5 * desc_norm + 0.5 * content_norm
    combined = alpha * cf_norm + beta * text_norm + gamma * emb_norm + delta * sent_norm

    for sid in seed_game_ids:
        if sid in combined.index:
            combined.loc[sid] = -1

    recs = combined.sort_values(ascending=False).head(topk)
    return pd.DataFrame({
        "game_id": recs.index,
        "score": recs.values,
        "name": [df_meta.loc[g, "name"] if g in df_meta.index else "" for g in recs.index],
    })

In [ ]:
def recommend_from_seeds(seed_ids, user_seen_set, k=20):
    """
    seed_ids: list[int]   (BGG ids)
    user_seen_set: set[int]
    returns: list[int] recommended BGG ids length <= k
    """
    seed_set = set(seed_ids)

    # over-generate then filter (important because hybrid_recommend doesn't know user history)
    rec_df = hybrid_recommend(seed_ids, topk=k + 300)

    if rec_df is None or len(rec_df) == 0:
        return []

    recs = rec_df["game_id"].tolist()

    filtered = []
    for gid in recs:
        if gid in seed_set:
            continue
        if gid in user_seen_set:
            continue
        filtered.append(int(gid))
        if len(filtered) >= k:
            break

    return filtered

In [ ]:
KS = [5, 10, 20]
SEED_SIZE = 3            # use 3 liked games as seeds
MAX_USERS = 2000         # cap for speed; increase if fast
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

# ensure valid
assert MIN_RELEVANT_FOR_EVAL >= SEED_SIZE + 1, "MIN_RELEVANT_FOR_EVAL must be >= SEED_SIZE+1"

users_sample = rng.permutation(eval_users)[:MAX_USERS]
print("Evaluating users:", len(users_sample))

In [ ]:
rows = []

for u in users_sample:
    rel_items = list(relevant_by_user[u])
    if len(rel_items) < SEED_SIZE + 1:
        continue

    rng.shuffle(rel_items)

    seeds = rel_items[:SEED_SIZE]
    holdout = set(rel_items[SEED_SIZE:])
    #holdout = set(rel_items[SEED_SIZE:SEED_SIZE+10])
    
    if len(holdout) == 0:
        continue

    seen = seen_by_user.get(u, set())

    recs = recommend_from_seeds(seeds, seen, k=max(KS))
    if len(recs) == 0:
        continue

    # relevance list for ndcg
    binary_rels = [1 if gid in holdout else 0 for gid in recs]

    row = {
        "user": u,
        "n_seen": len(seen),
        "n_relevant": len(rel_items),
        "seed_size": len(seeds),
        "holdout_size": len(holdout),
    }

    for k in KS:
        row[f"Recall@{k}"] = recall_at_k(recs, holdout, k)
        row[f"NDCG@{k}"] = ndcg_at_k(binary_rels, k)

    rows.append(row)

eval_df = pd.DataFrame(rows)
print("Evaluated rows:", len(eval_df))
eval_df.head()

In [ ]:
metric_cols = [c for c in eval_df.columns if c.startswith(("Recall@", "NDCG@"))]
summary = eval_df[metric_cols].mean(numeric_only=True)

print("=== Overall Summary ===")
display(summary)

# cold-start definition (few ratings total)
USER_COLD_N = 10
user_counts = ratings_df.groupby(USER_COL)[ITEM_COL].count().to_dict()

eval_df["is_user_cold"] = eval_df["user"].map(lambda x: user_counts.get(x, 0) < USER_COLD_N)

cold_summary = eval_df.groupby("is_user_cold")[metric_cols].mean(numeric_only=True)

print("=== Cold vs Non-cold Summary ===")
display(cold_summary)

# optional: show counts
display(eval_df["is_user_cold"].value_counts(dropna=False))

In [ ]:
out_path = "../Reports/eval_metrics_results.csv"
eval_df.to_csv(out_path, index=False)
print("Saved:", out_path)